In [1]:
import yfinance as yf
import pandas as pd 
import numpy as np

In [2]:
def pobierz_dane_z_rynku(waluta):
    ticker= yf.Ticker(waluta)
    dane= ticker.history(period="2y")
    return dane[['Close']]

In [5]:
dane_usd= pobierz_dane_z_rynku('USDPLN=X')
dane_eur= pobierz_dane_z_rynku('EURPLN=X')
dane_chf= pobierz_dane_z_rynku('CHFPLN=X')

dane_eur.head()

,Close
Date,
2024-03-20 00:00:00+00:00,4.31306
2024-03-21 00:00:00+00:00,4.30929
2024-03-22 00:00:00+00:00,4.30120
2024-03-25 00:00:00+00:00,4.31910
2024-03-26 00:00:00+00:00,4.30250


In [20]:
def polacz_i_oczysc_dane(dane_usd, dane_eur, dane_chf):
    dane_usd= dane_usd.rename(columns={'Close': 'Kurs_USD'})
    dane_eur= dane_eur.rename(columns={'Close': 'Kurs_EUR'})
    dane_chf= dane_chf.rename(columns={'Close': 'Kurs_CHF'})

    polaczone_usd_eur= pd.merge(dane_usd, dane_eur, left_index=True, right_index=True, how='outer')
    polaczone_wszystkie= pd.merge(polaczone_usd_eur, dane_chf, left_index=True, right_index=True, how='outer')

    polaczone_czyste= polaczone_wszystkie.dropna()
    polaczone_czyste.reset_index(inplace=True)
    polaczone_czyste['Date']= polaczone_czyste['Date'].dt.tz_localize(None)
    return polaczone_czyste



In [21]:
dane_gotowe= polacz_i_oczysc_dane(dane_usd, dane_eur, dane_chf)
dane_gotowe.tail()

,Date,Kurs_USD,Kurs_EUR,Kurs_CHF
513,2026-03-16,3.73146,4.26805,4.72276
514,2026-03-17,3.70450,4.26028,4.70071
515,2026-03-18,3.68979,4.25784,4.69920
516,2026-03-19,3.72680,4.27387,4.70422
517,2026-03-20,3.69060,4.27277,4.68562


In [24]:
def wylicz_wskazniki_ryzyka(tabela, okno=30):
    waluty= ['USD', 'EUR', 'CHF']
    for i in waluty:
        kurs= f'Kurs_{i}'
        #1. Średnia krocząca
        tabela[f'SMA_{okno}_{i}']= tabela[kurs].rolling(window=okno).mean()
        #2. Zwroty logarytmiczne
        tabela[f'Zwrot_{i}']= np.log(tabela[kurs] / tabela[kurs].shift(1))
        #3. Zmienność (standardowo 252 dni w roku, pomijamy weekendy i święta)
        tabela[f'Zmiennosc_{i}']= tabela[f'Zwrot_{i}'].rolling(window=okno).std()* np.sqrt(252)
        #4. VaR 95%
        tabela[f'VaR_95%_{i}']= tabela[f'Zwrot_{i}'].rolling(window=okno).quantile(0.05)
    
    return tabela


In [ ]:
dane_final= wylicz_wskazniki_ryzyka(dane_gotowe)
dane_final.head()

,Date,Kurs_USD,Kurs_EUR,Kurs_CHF,SMA_30_USD,Zwrot_USD,Zmiennosc_USD,VaR_95%_USD,SMA_30_EUR,Zwrot_EUR,Zmiennosc_EUR,VaR_95%_EUR,SMA_30_CHF,Zwrot_CHF,Zmiennosc_CHF,VaR_95%_CHF
0,2024-03-20,3.969000,4.31306,4.468120,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2024-03-21,3.940138,4.30929,4.448590,NaN,-0.007298,NaN,NaN,NaN,-0.000874,NaN,NaN,NaN,-0.004381,NaN,NaN
2,2024-03-22,3.960600,4.30120,4.412644,NaN,0.005180,NaN,NaN,NaN,-0.001879,NaN,NaN,NaN,-0.008113,NaN,NaN
3,2024-03-25,3.996484,4.31910,4.451770,NaN,0.009019,NaN,NaN,NaN,0.004153,NaN,NaN,NaN,0.008828,NaN,NaN
4,2024-03-26,3.969883,4.30250,4.415090,NaN,-0.006678,NaN,NaN,NaN,-0.003851,NaN,NaN,NaN,-0.008273,NaN,NaN


In [29]:
def usun_puste_dni_i_zaokraglij(dane_final):
    dane_final= dane_final.dropna().round(4)
    return dane_final

In [30]:
dane_final= usun_puste_dni_i_zaokraglij(dane_final)
dane_final.head()

C:\Users\pirek\AppData\Local\Temp\ipykernel_8744\852884812.py:2: UserWarning: obj.round has no effect with datetime, timedelta, or period dtypes. Use obj.dt.round(...) instead.
  dane_final= dane_final.dropna().round(4)


,Date,Kurs_USD,Kurs_EUR,Kurs_CHF,SMA_30_USD,Zwrot_USD,Zmiennosc_USD,VaR_95%_USD,SMA_30_EUR,Zwrot_EUR,Zmiennosc_EUR,VaR_95%_EUR,SMA_30_CHF,Zwrot_CHF,Zmiennosc_CHF,VaR_95%_CHF
30,2024-05-01,4.0596,4.3302,4.4143,4.0018,0.0085,0.1058,-0.0085,4.3029,0.0040,0.0680,-0.0057,4.4122,-0.0011,0.0940,-0.0087
31,2024-05-02,4.0334,4.3250,4.4028,4.0049,-0.0065,0.1052,-0.0085,4.3034,-0.0012,0.0680,-0.0057,4.4107,-0.0026,0.0935,-0.0087
32,2024-05-03,4.0335,4.3279,4.4364,4.0074,0.0000,0.1044,-0.0085,4.3043,0.0007,0.0678,-0.0057,4.4115,0.0076,0.0932,-0.0087
33,2024-05-06,4.0132,4.3173,4.4322,4.0079,-0.0050,0.1025,-0.0085,4.3043,-0.0025,0.0671,-0.0057,4.4109,-0.0009,0.0896,-0.0087
34,2024-05-07,3.9980,4.3047,4.4092,4.0089,-0.0038,0.1012,-0.0085,4.3043,-0.0029,0.0667,-0.0057,4.4107,-0.0052,0.0876,-0.0080


In [ ]:
def zapisz_do_powerbi(tabela):
    tabela.to_csv('dane_walutowe_do_powerbi.csv', index=False)